In [2]:

import os.path as op

import yaml
from FAM.processing import load_exp_settings, preproc_mridata, preproc_behdata
from FAM.fitting.prf_model import pRF_model
from FAM.fitting.decoding_model import Decoding_Model

from FAM.visualize.decoder_viewer import DecoderViewer

# load settings from yaml
with open('exp_params.yml', 'r') as f_in:
    params = yaml.safe_load(f_in)

In [3]:
sj = 'all'
system_dir = 'local'
exclude_sj = ['002']
wf_dir = None
use_atlas = None
prf_model_name = 'gauss'
fit_hrf = True
hemisphere = 'BH'
fa_model_name = 'glmsingle'

In [ ]:
FAM_data = load_exp_settings.MRIData(params, sj, 
                                    repo_pth = op.split(load_exp_settings.__file__)[0], 
                                    base_dir = system_dir, exclude_sj = exclude_sj, wf_dir = wf_dir)

print('Subject list is {l}\n'.format(l=str(FAM_data.sj_num)))



In [5]:
## Load preprocessing class for each data type ###

# get behavioral info 
FAM_beh = preproc_behdata.PreprocBeh(FAM_data)
# and mri info
FAM_mri = preproc_mridata.PreprocMRI(FAM_data)

## load pRF model class
FAM_pRF = pRF_model(FAM_data, use_atlas = use_atlas)

# set specific params
FAM_pRF.model_type['pRF'] = prf_model_name
FAM_pRF.fit_hrf = fit_hrf

## load FA Decoding model class
FAM_Decoder = Decoding_Model(FAM_data, use_atlas = use_atlas, pRFModelObj = FAM_pRF)

# make list of hemispheres to be fitted (useful for when using giftis)
hemis2fit = [hemisphere]
if FAM_data.sj_space in ['fsnative', 'fsaverage'] and hemisphere == 'BH':
    hemis2fit = ['hemi-L', 'hemi-R']
nifti_bool = False

In [ ]:
# get all participant bar positions for FA task
group_bar_pos_df = FAM_beh.get_group_FA_bar_position_dict(participant_list = FAM_data.sj_num, 
                                                        ses_num = None, 
                                                        ses_type = 'func', 
                                                        run_num = None)  

In [ ]:
group_bar_pos_df

In [ ]:
## get prf bar position dict
# to mask out FA trials that were not fully visible
prf_bar_coords_dict = FAM_beh.get_pRF_masked_bar_coords(participant_list = FAM_data.sj_num, 
                                                        ses = 'mean')

In [9]:
## load plotter class
plotter = DecoderViewer(FAM_data, DecoderObj = FAM_Decoder, use_atlas = use_atlas)

In [ ]:
## inputs for viz function

participant_list = FAM_data.sj_num
ROI_list = ['V1','V2','V3','hV4','V3AB','LO']
model_type = 'dog_hrf' #'gauss_hrf' #
ses = 'mean'
prf_file_ext =  FAM_mri.get_mrifile_ext(nifti_file = True)['pRF'] 
fa_file_ext = '_cropped.nii.gz'
mask_bool_df = FAM_beh.get_pRF_mask_bool(ses_type = 'func',
                                        crop_nr = FAM_data.task_nr_cropTR['pRF'], 
                                        shift = FAM_data.shift_TRs_num)
stim_on_screen = FAM_beh.get_stim_on_screen(task = 'pRF', 
                                            crop_nr = FAM_data.task_nr_cropTR['pRF'], 
                                            shift = FAM_data.shift_TRs_num)

In [11]:
### within viz function

In [ ]:
## load participant data keys
# for reference later on
data_keys_dict = FAM_Decoder.MRIObj.beh_utils.get_data_keys_dict(participant_list = participant_list, 
                                                                group_bar_pos_dict = group_bar_pos_df)

## get downsampled FA DM for group
lowres_DM_dict = FAM_Decoder.load_group_DM_dict(participant_list = participant_list, 
                                                group_bar_pos_df = group_bar_pos_df, 
                                                data_keys_dict = data_keys_dict,
                                                prf_bar_coords_dict = prf_bar_coords_dict)

## get FA bar position dict, across runs, for group
FA_run_position_df = FAM_beh.get_FA_group_run_position_df(participant_list = participant_list,
                                                          ses_type = 'func')

## reduce to minimum info necessary
FA_run_position_df = FAM_beh.squeeze_FA_bar_pos_df(FA_run_position_df = FA_run_position_df)

## get the dataframe with the behavioral outcomes
# for group
df_FA_beh_RT = FAM_beh.get_FA_RT(ses_type = 'func')

In [13]:
## get all runs reconstructed stim 
# for each ROI
# of all participants

ROIs_reconstructed_stim_dict = {}

# iterate over ROIs             
for roi_name in ROI_list:
    
    ## load reconstructed stim
    ROIs_reconstructed_stim_dict[roi_name] = FAM_Decoder.load_group_decoded_stim_dict(participant_list = participant_list, 
                                                                       roi_name = roi_name,
                                                                        model_type = model_type, 
                                                                       data_keys_dict = data_keys_dict)
    

In [ ]:
## PLOT ##
## correlate average reconstructed stim (all trials) with downsampled DM

plotter.plot_ground_truth_correlations(participant_list = participant_list, 
                                        reconstructed_stim_dict = ROIs_reconstructed_stim_dict, 
                                        lowres_DM_dict = lowres_DM_dict, 
                                        data_keys_dict = data_keys_dict,
                                        ROI_list = ROI_list, 
                                        mask_nan = True,
                                        model_type = model_type,
                                        return_df = False,
                                        fig_type = 'png')

In [ ]:
## get all possible trial combinations
# to use for bookkeeping of single trial DM
trial_combinations_df = FAM_Decoder.get_single_trial_combinations()

## average across runs
group_stim_dict, group_refDM_dict = FAM_Decoder.get_run_avg_stim_glmsing_trials(participant_list = participant_list, 
                                                                                ROI_list = ROI_list, 
                                                                                FA_run_position_df = FA_run_position_df, 
                                                                                trial_combinations_df = trial_combinations_df, 
                                                                                reconstructed_stim_dict = ROIs_reconstructed_stim_dict, 
                                                                                data_keys_dict = data_keys_dict,
                                                                                lowres_DM_dict = lowres_DM_dict)


In [ ]:
## PLOT ##

## get correlation of across run average

avg_stim_corr_df = plotter.plot_runAVG_ground_truth_correlations(participant_list = participant_list, 
                                                                group_stim_dict = group_stim_dict, 
                                                                group_refDM_dict = group_refDM_dict,
                                                                ROI_list = ROI_list, 
                                                                model_type = model_type,
                                                                return_df = True,
                                                                fig_type = 'svg')


In [ ]:
import scipy

## t test to check if group average corr significantly different from 0

for roi_name in ROI_list:
    t, p = scipy.stats.ttest_1samp(avg_stim_corr_df[avg_stim_corr_df['ROI'] == roi_name]['corr'].values, 
                                   popmean=0, alternative='greater')
    print('{:} t = {:.5f}, p = {:.5f}'.format(roi_name, t, p))
    
    if p < (0.001/len(ROI_list)):
        print('correct sig level ***')
    elif p < (0.01/len(ROI_list)):
        print('correct sig level **')
    elif p < (0.05/len(ROI_list)):
        print('correct sig level *')
    else:
        print('when corrected, not significant')


In [ ]:
## PLOT ##

# make single trial recontructed stim movie
# to inspect values

## average across participants
plotter.plot_trial_stim_movie(participant_list = participant_list, 
                              ROI_list = ROI_list, 
                              model_type = model_type, 
                              group_stim_dict = group_stim_dict, 
                              group_refDM_dict = group_refDM_dict, 
                              avg_pp = True, 
                              fig_type = 'mp4',
                              cmap = 'plasma', 
                              annot = False, 
                              interval = 132, 
                              figsize = (8,5), 
                              fps = 6, 
                              dpi = 200,
                              mask_edges = True)

# ## for each participant
# plotter.plot_trial_stim_movie(participant_list = participant_list, 
#                               ROI_list = ROI_list, 
#                               model_type = model_type, 
#                               group_stim_dict = group_stim_dict, 
#                               group_refDM_dict = group_refDM_dict, 
#                               avg_pp = False, 
#                               fig_type = 'mp4',
#                               cmap = 'magma', 
#                               annot = False, 
#                               interval = 132, 
#                               figsize = (8,5), 
#                               fps = 6, 
#                               dpi = 100,
#                               mask_edges = True)

In [ ]:
## get attention gradient values per pixel

# so per ROI reconstructed stim, get value of pixel for when there was an attended/unattended bar there
# (overlap? --> make optional to include/exclude from values)

ROI_pixel_df = FAM_Decoder.get_group_pix_intensity(participant_list = participant_list, 
                                                   ROI_list = ROI_list,
                                                   reconstructed_stim_dict = ROIs_reconstructed_stim_dict, 
                                                   lowres_DM_dict = lowres_DM_dict, 
                                                   data_keys_dict = data_keys_dict,
                                                   overlap_only = False)

## remake bar position df (unsqueezed)
FA_run_position_df = FAM_beh.get_FA_group_run_position_df(participant_list = participant_list,
                                                          ses_type = 'func')

## merge behav info and pixel intensity df

# first add bar positions
new_ROI_pixel_df = ROI_pixel_df.merge(FA_run_position_df, on=['sj', 'ses', 'run', 'trial_ind', 'bar_type'])
# then RT
new_ROI_pixel_df = new_ROI_pixel_df.merge(df_FA_beh_RT, on=['sj', 'run', 'ses', 'trial_ind'])

In [ ]:
## also get df with only overlap, to check
ROI_overlap_pixel_df = FAM_Decoder.get_group_pix_intensity(participant_list = participant_list, 
                                                   ROI_list = ROI_list,
                                                   reconstructed_stim_dict = ROIs_reconstructed_stim_dict, 
                                                   lowres_DM_dict = lowres_DM_dict, 
                                                   data_keys_dict = data_keys_dict,
                                                   overlap_only = True)

In [19]:
## get coordinates where there was overlap
## and filter out edges from pixel df 
# (edges will be more noisy, so no point in including in analysis)

new_ROI_pixel_df = new_ROI_pixel_df[new_ROI_pixel_df['coord_ind'].isin(ROI_overlap_pixel_df.coord_ind.unique())]

## and select only correct trials

new_ROI_pixel_df = new_ROI_pixel_df[new_ROI_pixel_df['correct'] == 1]


In [ ]:
## PLOT ###

# most plots should be created via this function
# so this is the main call of future script

plotter.plot_group_pixel_results(pixel_df = new_ROI_pixel_df, 
                               ROI_list = ROI_list, 
                               error_bars = 'within', 
                               figsize=(8,5), 
                               model_type = model_type, 
                               fig_type = 'pdf')

In [ ]:
### PLOT (in notebook) ###

## first make plot with attend vs unattend pixel values
# across ROIs

plotter.barplot_mean_pix_intensity(pixel_df = new_ROI_pixel_df, 
                                   ROI_list = ROI_list, 
                                   error_bars = 'within', 
                                   figsize=(8,5), 
                                   filename = None,
                                   point_color = '#f5007b',
                                   ylim2 = [0,.04],
                                   ylim = [0,.2])


In [ ]:
## test in repeated measures anova

from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm, AnovaRM

pixel_anova = AnovaRM(new_ROI_pixel_df, 
                depvar='intensity',
                subject='sj',
                within=['bar_type', 'ROI'],
                aggregate_func='mean'
                ).fit()
print(pixel_anova)

In [ ]:
## calculate ANOVA by removing each ROI
# we see that removing V3AB removes interaction (so mainly driven by V3AB)

for rname in ROI_list:

    print(rname)

    pixel_anova = AnovaRM(new_ROI_pixel_df[new_ROI_pixel_df['ROI'] != rname], 
                    depvar='intensity',
                    subject='sj',
                    within=['bar_type', 'ROI'],
                    aggregate_func='mean'
                    ).fit()
    print(pixel_anova)

In [ ]:
### PLOT (in notebook) ###

## make plot with attend vs unattend pixel values
# across ROIs and per bar configuration

plotter.pointplot_mean_bar_configuration(pixel_df = new_ROI_pixel_df, 
                                       ROI_list = ROI_list, 
                                       error_bars = 'within', 
                                       figsize=(10,5), 
                                       filename = None,
                                       show_diff = True)


In [ ]:
## check if bar configuration leads to differences in attention effect
# across ROIs

pixel_anova = AnovaRM(new_ROI_pixel_df, #[(new_ROI_pixel_df['ROI'] != 'V3AB')], 
                    depvar='intensity',
                    subject='sj',
                    within=['bar_type', 'bars_pos', 'ROI'],
                    aggregate_func='mean'
                    ).fit()
print(pixel_anova)



In [ ]:
## get average attention effect for given conditions
diff_config_df = plotter.DecoderObj.get_avg_attDiff(pixel_df = new_ROI_pixel_df, 
                                                    conditions = ['ROI', 'bars_pos'])

diff_config_df

In [ ]:
## check if bar configuration leads to differences in attention effect
# across ROIs

pixel_anova = AnovaRM(diff_config_df, 
                    depvar='att_diff',
                    subject='sj',
                    within=['bars_pos', 'ROI'],
                    aggregate_func='mean'
                    ).fit()
print(pixel_anova)

In [ ]:
## compare bars orientation

## calculate ANOVA per ROI
for rname in ROI_list:

    print(rname)

    pixel_anova = AnovaRM(diff_config_df[diff_config_df['ROI'] == rname], 
                    depvar='att_diff',
                    subject='sj',
                    within=['bars_pos'],
                    aggregate_func='mean'
                    ).fit()
    print(pixel_anova)

In [ ]:
## calculate ANOVA by removing each ROI
# we see that removing V3AB removes interaction (so mainly driven by V3AB)

for rname in ROI_list:

    print(rname)

    pixel_anova = AnovaRM(new_ROI_pixel_df[new_ROI_pixel_df['ROI'] != rname], 
                    depvar='intensity',
                    subject='sj',
                    within=['bar_type', 'bars_pos', 'ROI'],
                    aggregate_func='mean'
                    ).fit()
    print(pixel_anova)

In [ ]:
## compare bars orientation

## calculate ANOVA per ROI
for rname in ROI_list:

    print(rname)

    pixel_anova = AnovaRM(new_ROI_pixel_df[new_ROI_pixel_df['ROI'] == rname], 
                    depvar='intensity',
                    subject='sj',
                    within=['bar_type', 'bars_pos'],
                    aggregate_func='mean'
                    ).fit()
    print(pixel_anova)


In [34]:
# ## save pix intensity values to test in R

# # set path to save file
# pix_dir = op.join(plotter.MRIObj.derivatives_pth, 'pixel_intensity')
# os.makedirs(pix_dir, exist_ok=True)

# ## save accuracy per ROI
# for roi_name in ROI_list:
#     new_ROI_pixel_df[(new_ROI_pixel_df['ROI'] == roi_name)].to_csv(op.join(pix_dir, 'df_pix_%s.csv'%roi_name))
    


In [35]:

### NEW PLOTS, REFERENCED BY PIXEL ECC ###


In [25]:
### GROUP ECC INTO 3 RINGS ####
## add ring values to pix df

ring_pix_df = plotter.DecoderObj.group_ecc_rings(pixel_df = new_ROI_pixel_df)

## set ring colors
ring_ecc_colors = {1: '#e3855b', 2: '#f5d771', 3: '#7fb5b8'}

In [ ]:
## plot reference ring ecc

fig, _ = plotter.plot_pix_ecc_ref_heatmap(pixel_df = ring_pix_df, 
                                         cmap = 'Spectral', 
                                         desat = .8, 
                                         figsize = (8,5), 
                                         filename = None, 
                                         dpi = 100,
                                         ring_ecc = True)

In [ ]:
for roi_name in ROI_list:
    plotter.plot_ROI_pix_ringEccDist_barplot(ring_pix_df = ring_pix_df, 
                                            roi_name = roi_name,
                                            filename = None, 
                                            ylim = [0,.2], 
                                            ylim2 = [-.015,.04],
                                            figsize = (20,5), 
                                            error_bars =  'within', 
                                            showtitle = True, 
                                            showxlabel = True, 
                                            ring_ecc_colors = None, 
                                            point_color = '#FF0080')

In [ ]:
for roi_name in ROI_list:
    
    plotter.plot_ROI_pix_EccDist_barplot(pixel_df = new_ROI_pixel_df, 
                                        roi_name = roi_name,
                                        filename = None, 
                                        ylim = [0,.23], 
                                        ylim2 = [-.035,.065],
                                        figsize = (30,5), 
                                        error_bars =  'within', 
                                        showtitle = True, 
                                        showxlabel = True, 
                                        ecc_colors = None, 
                                        point_color = '#FF0080')

In [ ]:
### check for interaction between attention and pix ecc ###
## across distances

## calculate ANOVA per ROI
for rname in ROI_list:

    print(rname)

    pixel_anova = AnovaRM(ring_pix_df[ring_pix_df['ROI'] == rname], 
                    depvar='intensity',
                    subject='sj',
                    within=['bar_type', 'ring_ecc'],
                    aggregate_func='mean'
                    ).fit()
    print(pixel_anova)

In [37]:
# ## save pix intensity values to test in R

# # set path to save file
# pix_dir = op.join(plotter.MRIObj.derivatives_pth, 'pixel_intensity')
# os.makedirs(pix_dir, exist_ok=True)

# ## save accuracy per ROI
# for roi_name in ROI_list:
#     ring_pix_df[(ring_pix_df['ROI'] == roi_name)].to_csv(op.join(pix_dir, 'df_ring_pix_%s.csv'%roi_name))
    


In [ ]:
### Show the attention effect (so attended minus unattended) per ring ecc

plotter.plot_pix_ringEcc_AttDiff(ring_pix_df = ring_pix_df,  
                                  ROI_list = ROI_list, 
                                  error_bars = 'within', 
                                  fig_type = 'png',
                                  figsize=(15,3), 
                                  filename = None, 
                                  ylim = [-.005, .02])

In [36]:
#ring_pix_df